# 2018–2020 韩国关税署 10 月 1–20 日原稿附件

三份官方稿件页，每年最多一个按文件名固定选择的附件；源字节与 NAS 冷读先于内容解读。无 SOXX 结果、拟合或账户。

In [1]:
import hashlib,json,subprocess,tempfile
from datetime import datetime,timezone,date
from pathlib import Path
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from src.data.nas_market_data import publish_nas_snapshot,materialize_nas_snapshot
root=Path.cwd();card=root/'research/kcs_early20_2018_2020_oct_original_attachments_20261023';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest();assert plan['frozen'] and not (card/'run_state.json').exists()
parent=root/plan['parent_index']['card'];assert sha(parent/'summary.json')==plan['parent_index']['summary_sha256'] and json.loads((parent/'nas_binding.json').read_text())['verified']
raw=root/'data/snapshots/kcs_early20_2018_2020_oct_original_attachments_20261023';raw.mkdir(parents=True,exist_ok=True);assert not any(raw.iterdir())
state={'status':'started','plan_sha256':sha(card/'plan.json'),'GET_attempts':0,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];years=[];t=plan['transport']
def fetch(name,url):
    state['GET_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
    body=raw/name;headers=raw/(name+'.headers');started=datetime.now(timezone.utc).isoformat()
    cmd=['curl','--http1.1','--noproxy','*','--connect-timeout',str(t['connect_timeout_seconds']),'--max-time',str(t['total_timeout_seconds_each']),'--max-filesize',str(t['max_response_bytes_each']),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',t['user_agent'],'-w','%{http_code} %{url_effective}',url]
    try:
        run=subprocess.run(cmd,capture_output=True,text=True,timeout=40);parts=run.stdout.split(' ',1);rec={'name':name,'url':url,'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':int(parts[0]) if parts[0].isdigit() else None,'final_url':parts[1] if len(parts)>1 else None,'transport_error':run.stderr[:250]}
    except Exception as exc:rec={'name':name,'url':url,'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'transport_exception':type(exc).__name__+':'+str(exc)[:180]}
    rec.update(body_sha256=sha(body) if body.exists() else None,body_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None,headers_bytes=headers.stat().st_size if headers.exists() else 0)
    receipts.append(rec);(raw/'capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
    good=rec.get('curl_exit_code')==0 and rec.get('http_status')==200 and body.exists() and 0<body.stat().st_size<=t['max_response_bytes_each']
    return body if good else None
for entry in plan['exact_page_requests']:
    year=entry['year'];item={'year':year,'official_page_date':entry['display_date'],'newsId':entry['newsId'],'official_page_url':entry['url'],'status':'started'}
    page=fetch(f'{year}_page.html',entry['url'])
    if page is None:item['status']='page_transport_failure';years.append(item);continue
    soup=BeautifulSoup(page.read_bytes(),'html.parser');h=soup.select_one('h1');title=h.get_text(' ',strip=True) if h else ''
    full=soup.get_text(' ',strip=True)
    if entry['newsId'] not in page.read_text(errors='replace') or not all(q in title for q in ('10월','20일','수출입')) or '관세청' not in full or not any(q in full for q in (entry['display_date'],entry['display_date'].replace('-','.'))):
        item.update(status='page_identity_ambiguous',page_title=title);years.append(item);continue
    candidates=[]
    for link in soup.find_all('a'):
        href=link.get('href','');label=link.get_text(' ',strip=True);low=label.lower()
        if '/common/download.do' not in href or not any(low.endswith('.'+ext) for ext in ('hwp','hwpx','pdf')):continue
        if '수출입' not in label or not ('10월' in label or '10.1' in label) or '20' not in label:continue
        ext=low.rsplit('.',1)[-1];candidates.append({'label':label,'url':urljoin('https://www.korea.kr',href),'extension':ext})
    unique={x['url']:x for x in candidates};candidates=list(unique.values())
    preferred=[x for x in candidates if x['extension'] in ('hwp','hwpx')]
    selected=preferred if preferred else [x for x in candidates if x['extension']=='pdf']
    if len(selected)!=1:item.update(status='attachment_selection_ambiguous',page_title=title,candidates=candidates);years.append(item);continue
    choice=selected[0];body=fetch(f'{year}_attachment.{choice["extension"]}',choice['url'])
    if body is None:item.update(status='attachment_transport_failure',page_title=title,selected_attachment=choice);years.append(item);continue
    data=body.read_bytes();identity=(data.startswith(bytes.fromhex('D0CF11E0A1B11AE1')) if choice['extension']=='hwp' else (data.startswith(b'PK\x03\x04') if choice['extension']=='hwpx' else data.startswith(b'%PDF-')))
    item.update(status='source_bytes_received' if identity else 'attachment_signature_failure',page_title=title,selected_attachment=choice,attachment_sha256=sha(body),attachment_bytes=body.stat().st_size)
    years.append(item)
assert state['GET_attempts']<=plan['budget']['external_GETs_max']
manifest={'schema_version':1,'experiment':plan['experiment'],'plan_sha256':state['plan_sha256'],'capture_utc':datetime.now(timezone.utc).isoformat(),'receipts':receipts,'years':years,'historical_first_public_bytes_qualified':False}
(raw/'source_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2)+'\n');(card/'source_receipt.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2)+'\n')
files={str(p.relative_to(root)):'raw' for p in raw.iterdir() if p.is_file()}
try:
    binding=publish_nas_snapshot(workspace_root=root,source_manifest=raw/'source_manifest.json',files=files,as_of=date.today().isoformat(),source_metadata={'experiment':plan['experiment'],'source':'Korea government KCS 2018-2020 October original attachments'})
    (card/'nas_binding.json').write_text(json.dumps(binding,indent=2)+'\n')
    with tempfile.TemporaryDirectory(prefix='kcs-2018-2020-original-cold-') as temp:
        readback=materialize_nas_snapshot(binding,workspace_root=Path(temp))
        for obj in binding['objects']:assert sha(Path(temp)/obj['local_path'])==obj['sha256']
    assert binding['verified'] and readback['restored_objects']==len(files)
    nas_ok=True
except Exception as exc:nas_ok=False;binding=None;manifest['NAS_failure']=type(exc).__name__+':'+str(exc)[:180]
status='completed' if nas_ok else 'stopped';state.update(status=status,NAS_readback_verified=nas_ok);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
result={'decision':'official_KCS_2018_2020_Oct_original_attachment_identity_mapped_source_only' if nas_ok else 'needs_KCS_NAS_backup','GET_attempts':state['GET_attempts'],'years':years,'all_three_attachment_bytes_identified':all(x['status']=='source_bytes_received' for x in years),'NAS_source_id':binding['source_id'] if binding else None,'NAS_readback_verified':nas_ok,'historical_first_public_bytes_qualified':False,'full_monthly_training_panel_qualified':False,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'strategy_forward_days':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n');print(json.dumps({'decision':result['decision'],'GET_attempts':result['GET_attempts'],'years':years,'NAS_readback_verified':nas_ok},ensure_ascii=False))


{"decision": "official_KCS_2018_2020_Oct_original_attachment_identity_mapped_source_only", "GET_attempts": 6, "years": [{"year": 2018, "official_page_date": "2018-10-22", "newsId": "156299697", "official_page_url": "https://www.korea.kr/briefing/pressReleaseView.do?newsId=156299697", "status": "source_bytes_received", "page_title": "2018년 10월 1일 ∼ 10월 20일 수출입 현황", "selected_attachment": {"label": "181022보도자료+18년+10월+1일20일수출입현황.hwp", "url": "https://www.korea.kr/common/download.do?fileId=186280964&tblKey=GMN", "extension": "hwp"}, "attachment_sha256": "f15f03b2285ba0447c403da773e50c43ef5cf0988d2d551dc9400bfd6cd4d332", "attachment_bytes": 222208}, {"year": 2019, "official_page_date": "2019-10-21", "newsId": "156356469", "official_page_url": "https://www.korea.kr/briefing/pressReleaseView.do?newsId=156356469", "status": "source_bytes_received", "page_title": "’19년 10월 1일 ∼ 10월 20일 수출입 현황", "selected_attachment": {"label": "191021+19년+10월+1일++10월+20일+수출입+현황.hwp", "url": "https://www.korea.

# 2018–2020 年韩国关税署 10 月 1–20 日原稿附件

三个已定位的关税署官网页面（2018-10-22、2019-10-21、2020-10-21）各有一份按事前规则选中的 HWP 原稿。六次官网请求、三页与三附件的确切原始字节及收取时间、SHA-256 已进入 Yanxuan NAS，远端哈希和限定冷恢复通过。

按 [Hancom 官方 HWP5 记录结构](https://cdn.hancom.com/link/docs/%ED%95%9C%EA%B8%80%EB%AC%B8%EC%84%9C_%ED%8C%8C%EC%9D%BC%ED%98%95%EC%8B%9D_5.0_revision1.2.pdf) 做一次机械段落解析，先在 2021/2022 已验官方控制稿中准确找回半导体出口 **6,394／+23.9%** 与 **5,573／−12.8%**。同一解析在 2018–2020 原稿的出口摘要段落分别读到半导体同比 **+9.4%、−28.8%、+12.1%**；这三份捕获的 HWP **可解析正文**未呈现后来两年那种逐品类金额表，嵌入图片未做数值认证，因此**半导体出口金额保持未知**，不拿其他年份的同比或总出口额倒推。

这些稿件对总出口列出的当期工作日分别为 **14.5、13.5、12**（上年同段为 10、14.5、13.5）。工作日变化不能直接当作半导体自身按日调整系数。五个 10 月官方发布入口及 2018–2022 对应附件现可审计，但只构成五个年度的一个月份，且今日可取的附件字节没有独立首次公开／修订收据。它**不是完整历史训练面、合格 PIT 因子或 SOXX 超额证据**。本卡 0 SOXX 结果读取、0 拟合、0 账户；任何后续信号必须先决定只用官方报告同比还是另找原始金额，并补足每月版本与决策时钟。
